In [42]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pgmpy.models.BayesianNetwork import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination




In [43]:
users = pd.read_csv('users.dat', sep='::', engine='python', names=['user_id', 'gender', 'age', 'occupation', 'zip_code'], encoding='latin-1')
movies = pd.read_csv('movies.dat', sep='::', engine='python', names=['movie_id', 'title', 'genres'], encoding='latin-1')
ratings = pd.read_csv('ratings.dat', sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'], encoding='latin-1')


In [44]:
ratings.tail()

,user_id,movie_id,rating,timestamp
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648
1000208,6040,1097,4,956715569


In [45]:
users.head()

,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [46]:
unique_occupations = users['occupation'].unique()
print( unique_occupations)

if all((unique_occupations >= 0) & (unique_occupations <= 20)):
    print("true")
else:
    print("false")


[10 16 15  7 20  9  1 12 17  0  3 14  4 11  8 19  2 18  5 13  6]
true


In [47]:
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [48]:
print(users.isnull().sum())
print(movies.isnull().sum())
print(ratings.isnull().sum())


user_id       0
gender        0
age           0
occupation    0
zip_code      0
dtype: int64
movie_id    0
title       0
genres      0
dtype: int64
user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64


In [49]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')


In [50]:
users = users[['user_id', 'gender', 'age','occupation']]
movies = movies[['movie_id', 'genres', 'year']]
ratings = ratings[['user_id', 'movie_id', 'rating']]

In [51]:
movies.head()

,movie_id,genres,year
0,1,Animation|Children's|Comedy,1995
1,2,Adventure|Children's|Fantasy,1995
2,3,Comedy|Romance,1995
3,4,Comedy|Drama,1995
4,5,Comedy,1995


In [52]:
print(users.isnull().sum())
print(movies.isnull().sum())
print(ratings.isnull().sum())


user_id       0
gender        0
age           0
occupation    0
dtype: int64
movie_id    0
genres      0
year        0
dtype: int64
user_id     0
movie_id    0
rating      0
dtype: int64


In [53]:
print(users[users['age'] == 0])
print(users[users['gender'] == 0])
print(users[users["occupation"] == 0])


print(ratings[ratings['rating'] == 0])


print(movies[(movies['year'].isnull()) | (movies['year'] == '0')])
print(movies[movies['genres'] == 0])


Empty DataFrame
Columns: [user_id, gender, age, occupation]
Index: []
Empty DataFrame
Columns: [user_id, gender, age, occupation]
Index: []
      user_id gender  age  occupation
13         14      M   35           0
15         16      F   35           0
22         23      M   35           0
31         32      F   25           0
33         34      F   18           0
...       ...    ...  ...         ...
6009     6010      M   35           0
6018     6019      M   25           0
6022     6023      M   25           0
6030     6031      F   18           0
6038     6039      F   45           0

[711 rows x 4 columns]
Empty DataFrame
Columns: [user_id, movie_id, rating]
Index: []
Empty DataFrame
Columns: [movie_id, genres, year]
Index: []
Empty DataFrame
Columns: [movie_id, genres, year]
Index: []


In [54]:

movies = movies.assign(genres=movies['genres'].str.split('|')).explode('genres')

print(movies.head())

   movie_id      genres  year
0         1   Animation  1995
0         1  Children's  1995
0         1      Comedy  1995
1         2   Adventure  1995
1         2  Children's  1995


In [55]:
movies.head()

,movie_id,genres,year
0,1,Animation,1995
0,1,Children's,1995
0,1,Comedy,1995
1,2,Adventure,1995
1,2,Children's,1995


In [56]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6408 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  6408 non-null   int64 
 1   genres    6408 non-null   object
 2   year      6408 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 200.2+ KB


In [57]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 3 columns):
 #   Column    Non-Null Count    Dtype
---  ------    --------------    -----
 0   user_id   1000209 non-null  int64
 1   movie_id  1000209 non-null  int64
 2   rating    1000209 non-null  int64
dtypes: int64(3)
memory usage: 22.9 MB


In [58]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     6040 non-null   int64 
 1   gender      6040 non-null   object
 2   age         6040 non-null   int64 
 3   occupation  6040 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 188.9+ KB


In [59]:
weird_years = movies[(movies['year'] < 1900) | (movies['year'] > 2025)]
print(weird_years)



Empty DataFrame
Columns: [movie_id, genres, year]
Index: []


In [60]:
duplicate_movies = movies[movies.duplicated(subset='movie_id', keep=False)]
print(duplicate_movies)


      movie_id      genres  year
0            1   Animation  1995
0            1  Children's  1995
0            1      Comedy  1995
1            2   Adventure  1995
1            2  Children's  1995
...        ...         ...   ...
3876      3946      Action  2000
3876      3946       Drama  2000
3876      3946    Thriller  2000
3882      3952       Drama  2000
3882      3952    Thriller  2000

[4383 rows x 3 columns]


In [61]:
label_encoder = LabelEncoder()
users['gender'] = label_encoder.fit_transform(users['gender'])
print(users.head())

   user_id  gender  age  occupation
0        1       0    1          10
1        2       1   56          16
2        3       1   25          15
3        4       1   45           7
4        5       1   25          20


In [62]:
one_hot = pd.get_dummies(movies['genres'])

df_encoded = pd.concat([movies[['movie_id', 'year']], one_hot], axis=1)

movies = df_encoded.groupby(['movie_id', 'year']).max().reset_index()

movies.head()

,movie_id,year,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1995,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2,1995,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2,3,1995,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False
3,4,1995,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False
4,5,1995,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


In [63]:
full_data = ratings.merge(users, on='user_id', how='left')
full_data = full_data.merge(movies, on='movie_id', how='left')

train_data, test_data = train_test_split(full_data, test_size=0.2, random_state=42)

print(train_data.head())


        user_id  movie_id  rating  gender  age  occupation  year  Action  \
416292     2507      3035       2       1   25           4  1955   False   
683230     4087      2840       4       1    1           4  1999   False   
2434         19       457       3       1    1          10  1993    True   
688533     4118      2804       4       1   25           3  1983   False   
472584     2907       805       4       0   35           5  1996   False   

        Adventure  Animation  ...  Fantasy  Film-Noir  Horror  Musical  \
416292      False      False  ...    False      False   False    False   
683230      False      False  ...    False      False   False    False   
2434        False      False  ...    False      False   False    False   
688533      False      False  ...    False      False   False    False   
472584      False      False  ...    False      False   False    False   

        Mystery  Romance  Sci-Fi  Thriller    War  Western  
416292    False    False   False     

In [64]:
full_data.columns

Index(['user_id', 'movie_id', 'rating', 'gender', 'age', 'occupation', 'year',
       'Action', 'Adventure', 'Animation', 'Children's', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='object')

In [65]:



full_data['high_rating'] = full_data['rating'].apply(lambda x: 1 if x >= 4 else 0)

genre_columns = ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']  

selected_columns = ['user_id', 'age', 'gender', 'occupation', 'rating'] + genre_columns + ['high_rating']
final_data = full_data[selected_columns]



print(final_data.head())

   user_id  age  gender  occupation  rating  Action  Adventure  Animation  \
0        1    1       0          10       5   False      False      False   
1        1    1       0          10       3   False      False       True   
2        1    1       0          10       3   False      False      False   
3        1    1       0          10       4   False      False      False   
4        1    1       0          10       5   False      False       True   

   Children's  Comedy  ...  Film-Noir  Horror  Musical  Mystery  Romance  \
0       False   False  ...      False   False    False    False    False   
1        True   False  ...      False   False     True    False    False   
2       False   False  ...      False   False     True    False     True   
3       False   False  ...      False   False    False    False    False   
4        True    True  ...      False   False    False    False    False   

   Sci-Fi  Thriller    War  Western  high_rating  
0   False     False  False   

In [66]:
train_data, test_data = train_test_split(final_data, test_size=0.2, random_state=42)

train_data


,user_id,age,gender,occupation,rating,Action,Adventure,Animation,Children's,Comedy,...,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,high_rating
416292,2507,25,1,4,2,False,False,False,False,True,...,False,False,False,False,False,False,False,True,False,0
683230,4087,1,1,4,4,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,1
2434,19,1,1,10,3,True,False,False,False,False,...,False,False,False,False,False,False,True,False,False,0
688533,4118,25,1,3,4,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,1
472584,2907,35,0,5,4,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,1586,45,1,7,5,False,False,False,False,True,...,False,False,False,False,False,True,False,False,False,1
365838,2129,25,1,5,5,False,False,True,False,True,...,False,False,False,False,False,False,False,False,False,1
131932,854,25,0,16,3,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,0
671155,4033,25,1,6,5,False,True,False,False,False,...,False,False,False,False,True,False,False,False,False,1


In [67]:
train_data.columns

Index(['user_id', 'age', 'gender', 'occupation', 'rating', 'Action',
       'Adventure', 'Animation', 'Children's', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western',
       'high_rating'],
      dtype='object')

In [68]:
from pgmpy.models import DiscreteBayesianNetwork


In [69]:
target = train_data['high_rating']

features = train_data.drop(columns=['high_rating',"user_id"])

In [70]:
features

,age,gender,occupation,rating,Action,Adventure,Animation,Children's,Comedy,Crime,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
416292,25,1,4,2,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,True,False
683230,1,1,4,4,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2434,1,1,10,3,True,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
688533,25,1,3,4,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
472584,35,0,5,4,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,45,1,7,5,False,False,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
365838,25,1,5,5,False,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
131932,25,0,16,3,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
671155,25,1,6,5,False,True,False,False,False,False,...,True,False,False,False,False,True,False,False,False,False


In [71]:
movies.columns

Index(['movie_id', 'year', 'Action', 'Adventure', 'Animation', 'Children's',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [72]:
features.columns

Index(['age', 'gender', 'occupation', 'rating', 'Action', 'Adventure',
       'Animation', 'Children's', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance',
       'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='object')

In [73]:
genre_columns = ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
                 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
                 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

genre_counts = movies[genre_columns].sum()

print(genre_counts > 106)


Action          True
Adventure       True
Animation      False
Children's      True
Comedy          True
Crime           True
Documentary     True
Drama           True
Fantasy        False
Film-Noir      False
Horror          True
Musical         True
Mystery        False
Romance         True
Sci-Fi          True
Thriller        True
War             True
Western        False
dtype: bool


In [ ]:


 edges = [
    ('age', 'high_rating'),
    ('gender', 'high_rating'),
    ('occupation', 'high_rating'),
    ('rating', 'high_rating'),
    ('Action', 'high_rating'),
    ('Drama', 'high_rating'),
    ('Comedy', 'high_rating'),
    ("Children's", 'high_rating'),
    ('Animation', 'high_rating'),
    ('Adventure', 'high_rating'),
    ('Crime', 'high_rating'),
    ('Fantasy', 'high_rating'),
    ('Horror', 'high_rating'),
    ('Documentary', 'high_rating'),
    ('Musical', 'high_rating'),
    ('Romance', 'high_rating'),
    ('Sci-Fi', 'high_rating'),
    ('Thriller', 'high_rating'),
    ('War', 'high_rating'),
    ('Western', 'high_rating'),
    ('Mystery', 'high_rating'),
    ('Film-Noir', 'high_rating'),
]


model = DiscreteBayesianNetwork (edges)

full_data = features.copy()
full_data['high_rating'] = target

model.fit(full_data, estimator=MaximumLikelihoodEstimator)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'age': 'N', 'gender': 'N', 'occupation': 'N', 'rating': 'N', 'Action': 'N', 'Adventure': 'N', 'Animation': 'N', "Children's": 'N', 'Comedy': 'N', 'Crime': 'N', 'Documentary': 'N', 'Drama': 'N', 'Fantasy': 'N', 'Film-Noir': 'N', 'Horror': 'N', 'Musical': 'N', 'Mystery': 'N', 'Romance': 'N', 'Sci-Fi': 'N', 'Thriller': 'N', 'War': 'N', 'Western': 'N', 'high_rating': 'N'}


KeyboardInterrupt: 

In [ ]:
test_data.iloc[0]

user_id         5412
age               25
gender             1
occupation        12
rating             2
Action         False
Adventure      False
Animation      False
Children's     False
Comedy          True
Crime          False
Documentary    False
Drama          False
Fantasy        False
Film-Noir      False
Horror         False
Musical        False
Mystery        False
Romance        False
Sci-Fi         False
Thriller       False
War            False
Western        False
high_rating        0
Name: 895536, dtype: object

In [ ]:


infer = VariableElimination(model)

test_sample = {
    'age': 25,
    'gender': 1,
    'occupation':12,
    'Action': 1,
    'Drama': 0,
    'Comedy': 1,
    'Thriller': 0
}

prediction = infer.query(variables=['high_rating'], evidence=test_sample)

print(prediction)

+----------------+--------------------+
| high_rating    |   phi(high_rating) |
+================+====================+
| high_rating(0) |             0.4451 |
+----------------+--------------------+
| high_rating(1) |             0.5549 |
+----------------+--------------------+


In [ ]:
test_data.columns

Index(['user_id', 'age', 'gender', 'occupation', 'rating', 'Action',
       'Adventure', 'Animation', 'Children's', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western',
       'high_rating'],
      dtype='object')

In [ ]:
X_test = test_data.drop(columns=['high_rating',"user_id","rating"])


In [ ]:

predictions = []

    evidence = row.to_dict()
    prediction = infer.query(variables=['high_rating'], evidence=evidence)
    
    prob_high_rating = prediction.values[1]  
    
    pred_label = 1 if prob_high_rating > 0.5 else 0
    predictions.append(pred_label)

X_test['predicted_high_rating'] = predictions

print(X_test[['predicted_high_rating']])

c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\pgmpy\inference\ExactInference.py:771: SyntaxWarning: invalid escape sequence '\s'
  """


KeyboardInterrupt: 

In [ ]:
correct_predictions = (X_test['predicted_high_rating'] == test_data['high_rating']).sum()
total_predictions = len(X_test)

accuracy = correct_predictions / total_predictions

print(f" {correct_predictions}")
print(f" {total_predictions}")
print(f" {accuracy:.2f}")

 200041
 200042
 1.00
